# Тема 5/7. Архитектура рекомендательных систем
# Реализация сервиса рекомендаций
## Урок 4/5. Онлайн-рекомендации

### Загрузка данных

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

items = pd.read_parquet("data/preprocess/items.par")
events = pd.read_parquet("data/preprocess/events.par")
#######

# зададим точку разбиения для обучения кандидатов
train_test_global_time_split_date = pd.to_datetime("2017-08-01").date()
train_test_global_time_split_idx = events["started_at"] < train_test_global_time_split_date

events_train = events[train_test_global_time_split_idx] 
events_test = events[~train_test_global_time_split_idx]

#######
# Перекодирование
from sklearn.preprocessing import LabelEncoder

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = LabelEncoder()
item_encoder.fit(items["item_id"])
items["item_id_enc"] = item_encoder.transform(items["item_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train['item_id']) 
events_test["item_id_enc"] = item_encoder.transform(events_test['item_id'])  

/tmp/ipykernel_2175/604393583.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_train["item_id_enc"] = item_encoder.transform(events_train['item_id'])
/tmp/ipykernel_2175/604393583.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["item_id_enc"] = item_encoder.transform(events_test['item_id'])


In [3]:
user_encoder = LabelEncoder()
user_encoder.fit(events["user_id"])

events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

/tmp/ipykernel_2175/1125698526.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
/tmp/ipykernel_2175/1125698526.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])


In [4]:
from scipy import sparse
import sys

# создаём sparse-матрицу формата CSR 
user_item_matrix_train = sparse.csr_matrix((
    events_train["rating"],
    (events_train['user_id_enc'], events_train['item_id_enc'])),
    dtype=np.int8)

In [5]:
%env OPENBLAS_NUM_THREADS=1
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05, random_state=0)
als_model.fit(user_item_matrix_train)

env: OPENBLAS_NUM_THREADS=1


/home/mle-user/mle_projects/sprint-4/mle-recsys-start/.venv_mle_recsys_start/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mle-user/mle_projects/sprint-4/mle-recsys-start/.venv_mle_recsys_start/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 50/50 [03:12<00:00,  3.84s/it]


### Задание 1/6 - получить набор похожих объектов в similar_items
- Дополните код ниже, чтобы получить набор похожих объектов в similar_items. 
- Вы можете подглядеть решение в уроке «Коллаборативная фильтрация: ALS» 
    — там вы реализовывали похожую логику для получения персональных рекомендаций.

In [33]:
# получим энкодированные идентификаторы всех объектов, известных нам из events_train
train_item_ids_enc = events_train['item_id_enc'].unique()

max_similar_items = 10

# получаем списки похожих объектов, используя ранее полученную ALS-модель
# метод similar_items возвращает и сам объект, как наиболее похожий
# этот объект мы позже отфильтруем, но сейчас запросим на 1 больше
similar_items = als_model.similar_items(train_item_ids_enc, N=max_similar_items+1)

# преобразуем полученные списки в табличный формат
sim_item_item_ids_enc = similar_items[0]
sim_item_scores = similar_items[1]

similar_items = pd.DataFrame({
    "item_id_enc": train_item_ids_enc,
    "sim_item_id_enc": sim_item_item_ids_enc.tolist(), 
    "score": sim_item_scores.tolist()}) # ваш код здесь #)
similar_items = similar_items.explode(['sim_item_id_enc', 'score'], ignore_index=True)

# приводим типы данных
similar_items["sim_item_id_enc"] = similar_items["sim_item_id_enc"].astype('int') # ваш код здесь #
similar_items["score"] = similar_items["score"].astype("float")

# получаем изначальные идентификаторы
similar_items["item_id_1"] = item_encoder.inverse_transform(similar_items['item_id_enc']) # ваш код здесь #
similar_items["item_id_2"] = item_encoder.inverse_transform(similar_items['sim_item_id_enc']) # ваш код здесь #

similar_items = similar_items.drop(columns=["item_id_enc", "sim_item_id_enc"])

# убираем пары с одинаковыми объектами
similar_items = similar_items.query("item_id_1 != item_id_2")

similar_items

,score,item_id_1,item_id_2
1,0.922490,22034,22026
2,0.874765,22034,6882
3,0.873763,22034,22028
4,0.850654,22034,364089
5,0.835730,22034,9827
...,...,...,...
456209,0.534920,21847032,19904043
456210,0.515321,21847032,6167746
456211,0.507710,21847032,17908487
456212,0.496325,21847032,6349976


#### Укажите идентификатор объекта, наиболее похожего на объект 7126.

In [49]:
items.query('item_id == 7126')

,item_id,author,title,description,genre_and_votes,num_pages,average_rating,ratings_count,text_reviews_count,publisher,publication_year,country_code,language_code,format,is_ebook,isbn,isbn13,genre_and_votes_dict,genre_and_votes_str,item_id_enc
738212,7126,"Alexandre Dumas, Robin Buss",The Count of Monte Cristo,'On what slender threads do life and fortune h...,"{'Classics': 18271, 'Fiction': 6761, 'Historic...",1276,4.22,564778,12165,Penguin Classics,2003,US,eng,Paperback,False,0140449264,9780140449266,"{'Academic': None, 'Academic-Academia': None, ...","Classics 18271, Fiction 6761, Historical-Histo...",839


In [48]:
book_columns = ["author", "title", "publication_year", "average_rating", "genre_and_votes"]

similar_items_view = similar_items.query('item_id_1 == 7126').rename(columns={'item_id_2': 'item_id'})
similar_items_view = similar_items_view.merge(items.set_index('item_id')[book_columns], on=['item_id'])
similar_items_view

,score,item_id_1,item_id,author,title,publication_year,average_rating,genre_and_votes
0,0.948725,7126,7190,Alexandre Dumas,"The Three Musketeers (The D'Artagnan Romances,...",2001,4.06,"{'Classics': 9823, 'Fiction': 3256, 'Historica..."
1,0.940997,7126,24280,"Victor Hugo, Lee Fahnestock, Norman MacAfee",Les Misérables,1987,4.14,"{'Classics': 15159, 'Fiction': 5061, 'Historic..."
2,0.930144,7126,1953,"Charles Dickens, Richard Maxwell",A Tale of Two Cities,2003,3.82,"{'Classics': 20021, 'Fiction': 6969, 'Historic..."
3,0.925066,7126,58696,"Charles Dickens, Jeremy Tambling",David Copperfield,2004,3.96,"{'Classics': 7795, 'Fiction': 2831, 'Literatur..."
4,0.916340,7126,38296,James Fenimore Cooper,The Last of the Mohicans (The Leatherstocking ...,1982,3.69,"{'Classics': 2791, 'Fiction': 1314, 'Historica..."
5,0.916015,7126,2932,"Daniel Defoe, Virginia Woolf",Robinson Crusoe,2001,3.66,"{'Classics': 7725, 'Fiction': 3305, 'Adventure..."
6,0.913951,7126,7184,"Alexandre Dumas, David Coward, Auguste Maquet","Twenty Years After (The D'Artagnan Romances, #2)",1993,4.01,"{'Classics': 933, 'Fiction': 392, 'Historical-..."
7,0.911433,7126,387749,Lew Wallace,Ben-Hur: A Tale of the Christ,2007,4.02,"{'Classics': 884, 'Historical-Historical Ficti..."
8,0.909872,7126,7733,"Jonathan Swift, Robert DeMaria Jr.",Gulliver's Travels,2003,3.56,"{'Classics': 8077, 'Fiction': 3399, 'Fantasy':..."
9,0.909454,7126,30597,"Victor Hugo, Walter J. Cobb",The Hunchback of Notre-Dame,2001,3.97,"{'Classics': 6285, 'Fiction': 1974, 'Historica..."


#### Полученный набор вскоре вам снова понадобится, так что сохраните его в файле: 

In [50]:
path = 'data/online-recs/'
similar_items.to_parquet(path + "similar_items.parquet")

### Просмотр списка похожих объектов
- Полезно убедиться, что полученный набор действительно содержит похожие данные. 
- Например, можно оценить глазами списки похожих объектов для каких-то уже известных. 
- Создадим для этой цели функцию print_sim_items.

In [51]:
def print_sim_items(item_id, similar_items):

    item_columns_to_use = ["item_id", "author", "title", "genre_and_votes", "average_rating", "ratings_count"]
    
    item_id_1 = items.query("item_id == @item_id")[item_columns_to_use]
    display(item_id_1)
    
    si = similar_items.query("item_id_1 == @item_id")
    si = si.merge(items[item_columns_to_use].set_index("item_id"), left_on="item_id_2", right_index=True)
    display(si)

In [60]:
print_sim_items(17245, similar_items)

,item_id,author,title,genre_and_votes,average_rating,ratings_count
1058909,17245,"Bram Stoker, Nina Auerbach, David J. Skal",Dracula,"{'Classics': 19603, 'Horror': 10601, 'Fiction'...",3.98,636895


,score,item_id_1,item_id_2,author,title,genre_and_votes,average_rating,ratings_count
23937,0.928823,17245,480204,"Gaston Leroux, Alexander Teixeira de Mattos",The Phantom of the Opera,"{'Classics': 7010, 'Fiction': 2103, 'Horror': ...",3.97,144859
23938,0.900337,17245,51496,"Robert Louis Stevenson, Vladimir Nabokov, Merv...",The Strange Case of Dr. Jekyll and Mr. Hyde,"{'Classics': 12342, 'Fiction': 4037, 'Horror':...",3.79,229898
23939,0.898938,17245,93261,Washington Irving,The Legend of Sleepy Hollow,"{'Classics': 2594, 'Horror': 1182, 'Fiction': ...",3.74,26776
23940,0.897706,17245,295,Robert Louis Stevenson,Treasure Island,"{'Classics': 11249, 'Fiction': 4405, 'Adventur...",3.82,274424
23941,0.896470,17245,2623,"Charles Dickens, Marisa Sestino",Great Expectations,"{'Classics': 19645, 'Fiction': 6662, 'Literatu...",3.75,468462
23942,0.895993,17245,18254,"Charles Dickens, Philip Horne, Gerald Dickens",Oliver Twist,"{'Classics': 11450, 'Fiction': 3656, 'Historic...",3.85,235560
23943,0.886899,17245,7190,Alexandre Dumas,"The Three Musketeers (The D'Artagnan Romances,...","{'Classics': 9823, 'Fiction': 3256, 'Historica...",4.06,198892
23944,0.881911,17245,24213,"Lewis Carroll, John Tenniel, Martin Gardner",Alice's Adventures in Wonderland & Through the...,"{'Classics': 11568, 'Fantasy': 6184, 'Fiction'...",4.06,344482
23945,0.878392,17245,2932,"Daniel Defoe, Virginia Woolf",Robinson Crusoe,"{'Classics': 7725, 'Fiction': 3305, 'Adventure...",3.66,181415
23946,0.870232,17245,1953,"Charles Dickens, Richard Maxwell",A Tale of Two Cities,"{'Classics': 20021, 'Fiction': 6969, 'Historic...",3.82,646983
